<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
os.chdir("/content")
REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Rebuild the Week-4 baseline rule exactly, so it's scored on the SAME data
df["is_stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["is_visible"] = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = df["is_stale"] * df["is_visible"] * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(df.shape)

(30000, 48)


## 1. Method choice and why

**Question shape:** "which pages first?" — a ranking problem, per training-honest-models'
table. Start with **Logistic Regression** (readable), then **Random Forest** (stronger,
already shown in the starter pipeline to beat a hand rule 3.1x on Precision@50).

**Why these two:** Logistic Regression gives an interpretable coefficient-based view;
Random Forest is the reference model this whole track's starter pipeline already
validated as the strongest simple option (`outputs/model_results.json`: 0.740
Precision@50 vs. 0.240 baseline). Both output probabilities, which is what a ranking
task needs — not just a label.

**Label note:** still using the proxy `is_declining_label` (trend_direction-derived),
flagged since ML-02/03 as a current-window proxy, not an observed future outcome. This
is the established target for this week's exercise; a future-window label is the
Week-6+ direction.

In [2]:
candidate_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                       "avg_position", "ctr", "word_count", "engagement_rate", "scroll_rate"]
features = [c for c in candidate_features if c in df.columns]
print("Using features:", features)

Using features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate', 'scroll_rate']


## 2. Split design

**Grouped-by-client split.** Pages from the same client may share hidden characteristics
(site structure, content strategy) — a random split lets the model memorize
client-specific patterns and fake skill. This matches the starter pipeline's own
`client_holdout` strategy and the flyrank-data skill's recommendation to always group
by `client_id` for this kind of split.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

print(f"Train: {len(train_idx)} rows, {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test:  {len(test_idx)} rows, {df.iloc[test_idx]['client_id'].nunique()} clients")
overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print(f"Client overlap between train/test: {len(overlap)} (should be 0)")

Train: 19166 rows, 22 clients
Test:  10834 rows, 10 clients
Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

Training Logistic Regression and Random Forest on the grouped-split training set,
evaluating all three (baseline included) on the SAME held-out test rows, same metrics.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
base_test = df.iloc[test_idx]["baseline_score"].values
base_rate = y_test.mean()

logreg = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                             class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results = []
for name, scores in [("baseline (stale_visible_page)", base_test),
                      ("logistic_regression", logreg_scores),
                      ("random_forest", rf_scores)]:
    results.append({
        "method": name,
        "precision_at_20": precision_at_k(scores, y_test.values, 20),
        "precision_at_50": precision_at_k(scores, y_test.values, 50),
        "base_rate": base_rate,
    })

results_df = pd.DataFrame(results)
print(results_df.round(3))

                          method  precision_at_20  precision_at_50  base_rate
0  baseline (stale_visible_page)             0.65             0.64      0.559
1            logistic_regression             0.75             0.66      0.559
2                  random_forest             0.45             0.60      0.559


**Findings:** Logistic Regression wins clearly at both K (0.75 @20, 0.66 @50), beating
both the baseline and Random Forest. Random Forest actually underperforms — 0.45 @20 is
*below* the base rate (0.559), meaning at the top 20 it does worse than random guessing
would on this label. At @50 it recovers somewhat (0.60) but still trails Logistic
Regression. This is the opposite of the starter pipeline's own result (where RF beat
LogReg substantially) — likely because this proxy label (within-current-window
`is_declining_label`) and the grouped-by-client split behave differently than the
starter's evaluation setup. Simplicity won here: Logistic Regression's linear
combination of a few strong features outperformed a more complex ensemble on this
particular split.

## 4. Errors and interpretation

Looking at where the strongest model is wrong, what it leans on, and three concrete
hard cases.

In [5]:
# Top 3 features by importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Top features:")
print(importances.head(3))

# 3 concrete wrong cases: high predicted probability but label was actually 0 (false positives)
test_view = df.iloc[test_idx].copy()
test_view["rf_score"] = rf_scores
wrong = test_view[(test_view["rf_score"] > 0.7) & (test_view["is_declining_label"] == 0)]
print(f"\nHigh-confidence false positives: {len(wrong)}")
print(wrong[["content_id", "rf_score", "impressions_90d", "avg_position",
             "days_since_last_update", "trend_direction"]].head(3))

Top features:
impressions_90d     0.299866
avg_position        0.210113
content_age_days    0.179609
dtype: float64

High-confidence false positives: 1090
              content_id  rf_score  impressions_90d  avg_position  \
13  content_a5a2fbc76336  0.802155              307          39.8   
21  content_9d548144b06d  0.700383               86          12.6   
26  content_72c5c2d73e5a  0.717633             2426          30.0   

    days_since_last_update trend_direction  
13                     103          stable  
21                      20          stable  
26                      13          stable  


**Top features:** `impressions_90d` (0.30), `avg_position` (0.21), `content_age_days`
(0.18) — none suspiciously dominant, all plausible: high-traffic pages are more visible
to decline detection, worse position correlates with weaker performance, and older
content has had more time to drift. No single feature towers over the rest, which is a
good sign against leakage.

**Where the model is wrong:** the 1,090 high-confidence false positives mostly share a
pattern — `trend_direction == "stable"`, not "down". The model is confusing genuinely
stable pages for declining ones, likely because stable pages sit in a similar
mid-range of impressions/position/age as declining ones — the label boundary between
"stable" and "down" is where the signal is genuinely ambiguous, not a clear model flaw.

**Three hard cases:**
1. `content_a5a2fbc76336` — 307 impressions, position 39.8, updated 103 days ago,
   labeled stable. Weak position and moderate age look decline-like, but traffic held
   steady — a case where the features alone can't distinguish "quietly stable" from
   "starting to decline."
2. `content_9d548144b06d` — only 86 impressions, position 12.6, updated recently (20
   days), labeled stable. Very low traffic makes this a noisy case regardless of label;
   small samples are inherently hard to score confidently.
3. `content_72c5c2d73e5a` — 2,426 impressions, position 30.0, updated just 13 days ago,
   labeled stable. High traffic with a mid-tier position — likely why the model flagged
   it, but the recent update (13 days) is exactly the kind of signal a within-window
   proxy label doesn't capture well.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.